<a href="https://colab.research.google.com/github/abdullahnaeem151015-lgtm/ML-pipeline/blob/main/Copy_of_w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


For my lane, I first identify pages with high impressions and low CTR that meet my refresh rule. I then rank these pages so the strongest refresh opportunities appear first, allowing the team to know which pages should be reviewed first and why.

In [4]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import seaborn as sns
from huggingface_hub import hf_hub_download

print("Trying to download the warehouse dataset...")

# Hugging Face dataset details
repo_id = "FlyRank/internship-warehouse"
file_name = "fact_content_daily_performance_sample.parquet"

df_warehouse = pd.DataFrame() # Initialize df_warehouse
df_content = pd.DataFrame()   # Initialize df_content

try:
    # Download the file from Hugging Face
    local_file_path = hf_hub_download(
        repo_id=repo_id,
        filename=file_name,
        repo_type="dataset"
    )

    print("Dataset downloaded successfully.")
    print("Local file path:")
    print(local_file_path)

    # Read the parquet file, loading only the necessary columns
    required_columns = [
        'content_hash_id',
        'gsc_clicks',
        'gsc_impressions',
        'gsc_avg_position'
    ]
    df_warehouse = pd.read_parquet(local_file_path, columns=required_columns)

    print("\nDataset loaded successfully.")
    print("Shape:", df_warehouse.shape)

    print("\nFirst 10 rows:")
    display(df_warehouse.head(10))

    print("\nColumns:")
    print(df_warehouse.columns.tolist())

    # User wants daily granularity, so df_content should be df_warehouse with content_id renamed
    df_content = df_warehouse.copy()

    # Rename content_hash_id to content_id for consistency with subsequent cells
    df_content = df_content.rename(columns={'content_hash_id': 'content_id'})

    # Ensure numerical types and handle potential NaNs for calculation safety
    df_content['gsc_impressions'] = pd.to_numeric(df_content['gsc_impressions'], errors='coerce').fillna(0)
    df_content['gsc_clicks'] = pd.to_numeric(df_content['gsc_clicks'], errors='coerce').fillna(0)
    df_content['gsc_avg_position'] = pd.to_numeric(df_content['gsc_avg_position'], errors='coerce').fillna(0)

    # Calculate CTR: clicks / impressions
    # We'll handle cases where impressions are zero to avoid division by zero errors.
    df_content['ctr'] = df_content.apply(
        lambda row: row['gsc_clicks'] / row['gsc_impressions']
        if row['gsc_impressions'] > 0 else 0,
        axis=1
    )

    print(f"\nSuccessfully loaded {len(df_content)} daily content performance records with relevant columns for analysis.")

except Exception as e:
    print("\nSomething went wrong while loading the dataset.")
    print("Error:")
    print(e)

    print(
        "\nIf you see '401 Unauthorized' or 'gated repo', "
        "you probably need to log in to Hugging Face "
        "and provide an access token."
    )

Trying to download the warehouse dataset...


fact_content_daily_performance_sample.pa(…): reconstructing file:   0%|          |  0.00B /  145MB            

fact_content_daily_performance_sample.pa(…): downloading bytes:           |  0.00B            

Dataset downloaded successfully.
Local file path:
/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance_sample.parquet

Dataset loaded successfully.
Shape: (11694072, 4)

First 10 rows:


,content_hash_id,gsc_clicks,gsc_impressions,gsc_avg_position
0,content_1a6296faee432dae,0,0,NaN
1,content_73f21e612565035a,0,0,NaN
2,content_5a5be514ff559598,0,0,NaN
3,content_05b377d0c8a5cfd8,0,0,NaN
4,content_dc34c661d63e55a9,0,0,NaN
5,content_dcbfbaf912c88c76,0,0,NaN
6,content_44de3cf116b9e3f1,0,0,NaN
7,content_c3283758d93f34fd,0,0,NaN
8,content_859b5acb04908ae3,0,0,NaN
9,content_be99356ea2fc1df1,0,0,NaN



Columns:
['content_hash_id', 'gsc_clicks', 'gsc_impressions', 'gsc_avg_position']

Successfully loaded 11694072 daily content performance records with relevant columns for analysis.


In [5]:

##Rule
REASON_CODE_REFRESH_POTENTIAL_CTR_GAIN = 'REFRESH_POTENTIAL_CTR_GAIN'

# Calculate thresholds for 'low CTR' and 'high impressions'
# Since the 25th percentile for CTR can be 0, making the condition impossible to meet,
# let's redefine 'low CTR' more meaningfully.
# Option 1: Use a small, fixed positive value for CTR (e.g., 0.001)
# Option 2: Use the 25th percentile among items with CTR > 0
# Option 3: Use the 50th percentile as a general 'low' if 25th is 0, or just a custom value.

# Let's try Option 2: Calculate the 25th percentile of CTR for items that actually have clicks.
# If even that is zero, we'll fall back to a small positive number.

non_zero_ctrs = df_content[df_content['ctr'] > 0]['ctr']
if not non_zero_ctrs.empty:
    low_ctr_threshold = non_zero_ctrs.quantile(0.25)
    if low_ctr_threshold == 0:
        low_ctr_threshold = 0.0001
else:
    low_ctr_threshold = 0.0001

# And 'high impressions' to be above the 90th percentile of impressions_90d, only considering non-zero impressions.
# This ensures the threshold reflects actual high-traffic pages, not those with zero impressions.
non_zero_impressions = df_content[df_content['gsc_impressions'] > 0]['gsc_impressions']
if not non_zero_impressions.empty:
    high_impressions_threshold = non_zero_impressions.quantile(0.90)
else:
    # Fallback if all impressions are zero or no data
    high_impressions_threshold = 100 # Default to a reasonable high impression threshold

print(f"New Threshold for Low CTR (below {low_ctr_threshold:.4f})")
print(f"Threshold for High Impressions (above {high_impressions_threshold:.0f})")

# Identify content that meets the criteria
high_potential_refresh_items = df_content[
    (df_content['ctr'] < low_ctr_threshold) &
    (df_content['gsc_impressions'] > high_impressions_threshold)
].copy()

# Assign the reason code to these identified items
high_potential_refresh_items['reason_code'] = REASON_CODE_REFRESH_POTENTIAL_CTR_GAIN

print("\nExamples of High-Potential Refresh Opportunity items:")
display(
    high_potential_refresh_items[
        ['content_id', 'gsc_impressions', 'gsc_clicks', 'ctr', 'reason_code']
    ].head(10)
)

print(
    f"\nTotal items identified with '{REASON_CODE_REFRESH_POTENTIAL_CTR_GAIN}': "
    f"{len(high_potential_refresh_items)}"
)

New Threshold for Low CTR (below 0.0063)
Threshold for High Impressions (above 113)

Examples of High-Potential Refresh Opportunity items:


,content_id,gsc_impressions,gsc_clicks,ctr,reason_code
1967,content_e86d749e34db9b73,161,0,0.000000,REFRESH_POTENTIAL_CTR_GAIN
2053,content_8d4c42e5457c9e2e,207,0,0.000000,REFRESH_POTENTIAL_CTR_GAIN
8417,content_e4376489e7635039,193,0,0.000000,REFRESH_POTENTIAL_CTR_GAIN
8419,content_ec8d97984dbf4376,285,0,0.000000,REFRESH_POTENTIAL_CTR_GAIN
8424,content_ba0d2a28965016f9,733,2,0.002729,REFRESH_POTENTIAL_CTR_GAIN
8432,content_0163a234ae1e7e31,129,0,0.000000,REFRESH_POTENTIAL_CTR_GAIN
8433,content_bee0d8b86a9e4e21,713,1,0.001403,REFRESH_POTENTIAL_CTR_GAIN
8438,content_8ae43663243321d6,164,0,0.000000,REFRESH_POTENTIAL_CTR_GAIN
8441,content_c588cc6c0dce0f2b,234,0,0.000000,REFRESH_POTENTIAL_CTR_GAIN
8442,content_942a925e0c719a03,270,0,0.000000,REFRESH_POTENTIAL_CTR_GAIN



Total items identified with 'REFRESH_POTENTIAL_CTR_GAIN': 292021


In [6]:
#Create a copy so the original dataframe remains unchanged
ranked_queue = df_content.copy()

# ---------------------------------------------------------
# 1. Scale impressions and CTR to the same 0–1 range
# ---------------------------------------------------------

scaler = MinMaxScaler()

ranked_queue[
    ["impressions_scaled", "ctr_scaled"]
] = scaler.fit_transform(
    ranked_queue[["gsc_impressions", "ctr"]]
)

# ---------------------------------------------------------
# 2. Convert CTR into a low-CTR opportunity score
# Low CTR should receive a higher score
# ---------------------------------------------------------

ranked_queue["low_ctr_score"] = 1 - ranked_queue["ctr_scaled"]

# ---------------------------------------------------------
# 3. Calculate the multiplicative opportunity score
# A page must have both:
# - high impressions
# - low CTR
# ---------------------------------------------------------

ranked_queue["score"] = (
    ranked_queue["impressions_scaled"]
    * ranked_queue["low_ctr_score"]
)

# 4. Assigning action to each row based on its results
def assign_action(row):
    # Using the thresholds calculated in the previous cell for consistency
    if row["gsc_impressions"] > high_impressions_threshold and row["ctr"] < low_ctr_threshold:
        return "REVIEW_FOR_REFRESH"
    else:
        return "Review for general optimization potential"

# ---------------------------------------------------------
# 5. Displaying all the important columns that supports my ranked action
# ---------------------------------------------------------

ranked_queue["action"] = ranked_queue.apply(assign_action, axis=1)

# Initialize 'reason_code' column with None
ranked_queue['reason_code'] = None

# Assign REASON_CODE_REFRESH_POTENTIAL_CTR_GAIN where action is 'REVIEW_FOR_REFRESH'
ranked_queue.loc[
    ranked_queue['action'] == "REVIEW_FOR_REFRESH", 'reason_code'
] = REASON_CODE_REFRESH_POTENTIAL_CTR_GAIN

# Sort by score and then remove duplicate to get one row per content_id with the highest score
ranked_queue = ranked_queue.sort_values(by='score', ascending=False).drop_duplicates(subset=['content_id']).reset_index(drop=True)

display(ranked_queue[['content_id', 'ctr', 'gsc_impressions', 'score', 'action', 'reason_code']].head(10))

,content_id,ctr,gsc_impressions,score,action,reason_code
0,content_963de14b1f58978f,0.003620,245826,0.996380,REVIEW_FOR_REFRESH,REFRESH_POTENTIAL_CTR_GAIN
1,content_eadb33b5df496f4a,0.003504,49373,0.200142,REVIEW_FOR_REFRESH,REFRESH_POTENTIAL_CTR_GAIN
2,content_545bb6cc7081ded3,0.002329,48953,0.198673,REVIEW_FOR_REFRESH,REFRESH_POTENTIAL_CTR_GAIN
3,content_f88878f155e4838d,0.007169,45890,0.185338,Review for general optimization potential,None
4,content_0ec99ef7d7e11565,0.001632,30645,0.124458,REVIEW_FOR_REFRESH,REFRESH_POTENTIAL_CTR_GAIN
5,content_012de75c008aa653,0.000000,25191,0.102475,REVIEW_FOR_REFRESH,REFRESH_POTENTIAL_CTR_GAIN
6,content_ba462518dad435fc,0.000119,25111,0.102137,REVIEW_FOR_REFRESH,REFRESH_POTENTIAL_CTR_GAIN
7,content_b902320872acab45,0.000257,23358,0.094994,REVIEW_FOR_REFRESH,REFRESH_POTENTIAL_CTR_GAIN
8,content_11bf4c33adea7bdc,0.000000,20225,0.082274,REVIEW_FOR_REFRESH,REFRESH_POTENTIAL_CTR_GAIN
9,content_cc26620b2cbb837f,0.001293,20104,0.081676,REVIEW_FOR_REFRESH,REFRESH_POTENTIAL_CTR_GAIN


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


**Intended Use**
In most of the content teams primarily person who uses this rule for identifying pages that needs a refresh is SEO expert while the secondary user can be content manager or copywriter.

**Limits:**

* This rule can only suggests that page needs a review, it does not automatically tells that this page needs a guranteed refresh.

* Low CTR pages identified by this rule does not guarantee that the customer is not engaging with this page. There can be other reasons of low CTR like seasonality, overall low traffic for that particular content, page not showing on the top position, not spending enough budget on that particular content etc. So there needs a check on these factors before stating that this page has low ctr.

* If the new contents are introduced in the dataset my existing rule may not satisfy it   





## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


**Human Review**

Before acting on the rule of identifying the pages that needs a refresh person should check the internal factors that can be likely the reason of low CTR or high impressions like seasonality, search intent for that particular content, SERP features for a particular content, search position etc.

**No-Go list**

The things that should be never automated can be:

* Final decision on determining whether page needs a refresh

* Concluding that the low ctr is just because that customer is not engaging with the page

* Concluding that the content is of low quality just because of low ctr

* Ultimately, a "refresh" implies an update or improvement to the content itself. While metrics can flag potential issues, judging the actual quality, accuracy, comprehensiveness, and overall effectiveness of the content (and what specific changes are needed) is a highly human task that relies on domain expertise and editorial judgment.









## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


The rule should be re-evaluated under the reasons like if the CTR or impression thresholds change significantly over time, if the number of pages being flagged changes unusually,  if a large proportion of refreshed pages do not show CTR improvement, if rule perform differently across different seasons or if new contents are introduced for which my existing rule do not apply. These signals indicate that my recommendations might went stale

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


My following codes exports my queue and it's important figures to work/outputs section.

In [7]:
import os

# Create the outputs directory if it doesn't exist
output_dir = 'work/outputs/'
os.makedirs(output_dir, exist_ok=True)

# Export the calculated thresholds and the number of identified items
with open(os.path.join(output_dir, 'thresholds_and_counts.txt'), 'w') as f:
    f.write(f'Low CTR Threshold: {low_ctr_threshold:.4f}\n')
    f.write(f'High Impressions Threshold: {high_impressions_threshold:.0f}\n')
    f.write(f'Total items identified with \'{REASON_CODE_REFRESH_POTENTIAL_CTR_GAIN}\': {len(high_potential_refresh_items)}\n')

print(f"Thresholds and counts exported to {output_dir}thresholds_and_counts.txt")

Thresholds and counts exported to work/outputs/thresholds_and_counts.txt


In [9]:
# Export the ranked queue as a CSV file for further analysis in the paper
ranked_queue[['content_id', 'ctr', 'gsc_impressions', 'score', 'action', 'reason_code']].to_csv(os.path.join(output_dir, 'ranked_queue.csv'), index=False)

print(f"Ranked queue exported to {output_dir}ranked_queue.csv")

Ranked queue exported to work/outputs/ranked_queue.csv


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.